# Sheet 4.2: MLOps foundations: pipelines, tracking, tests, CI

**Roadmap topic 4.2** (Fanout 11.01-11.07, 11.21-11.22). Work through DataTalksClub MLOps Zoomcamp modules 1-4 and the testing chapters of *Made With ML* alongside this sheet.

The first exercises build small versions of DVC and MLflow yourself, so you understand what the real tools do. Exercise 5 then applies the real tools to your Step 3 code.

**Rules:** fill every `# YOUR CODE` cell, never edit `CHECK` cells. Runs on CPU.

In [ ]:
import os, json, time, hashlib, tempfile, shutil, math, numpy as np, torch, yaml

## Exercise 1: A content-addressed pipeline (what `dvc repro` does)

A pipeline is a list of stages, each `dict(name, fn, deps, outs, params)`. `fn(params)` reads `deps` files and writes `outs` files. Implement `run_pipeline(stages, workdir, lock_path)`:
- Order stages so every stage runs after the stages producing its deps (topological sort; raise `ValueError` on a cycle).
- A stage is **skipped** if the lock file records the same hash of (its deps' file contents, its params as sorted JSON) **and** its outs exist with their recorded hashes.
- Otherwise run it and update the lock. Return the list of stage names that actually ran.

In [ ]:
def file_hash(path):
    # YOUR CODE
    raise NotImplementedError
def run_pipeline(stages, workdir, lock_path):
    # YOUR CODE
    raise NotImplementedError

In [ ]:
# ---- CHECK 1 (do not edit) ----
def _check1():
    d = tempfile.mkdtemp(); P = lambda f: os.path.join(d, f)
    open(P("raw.txt"), "w").write("3 1 4 1 5 9 2 6")
    def prep(p): open(P("clean.txt"), "w").write(" ".join(str(int(x) * p["scale"]) for x in open(P("raw.txt")).read().split()))
    def train(p): open(P("model.json"), "w").write(json.dumps({"w": sum(map(int, open(P("clean.txt")).read().split())) * p["lr"]}))
    def evaluate(p): open(P("metrics.json"), "w").write(json.dumps({"score": json.load(open(P("model.json")))["w"] + 1}))
    st = lambda scale=2, lr=0.1: [dict(name="evaluate", fn=evaluate, deps=["model.json"], outs=["metrics.json"], params={}),
                                  dict(name="train", fn=train, deps=["clean.txt"], outs=["model.json"], params={"lr": lr}),
                                  dict(name="prep", fn=prep, deps=["raw.txt"], outs=["clean.txt"], params={"scale": scale})]
    L = P("lock.json")
    assert run_pipeline(st(), d, L) == ["prep", "train", "evaluate"], "wrong order or stages skipped on first run"
    assert run_pipeline(st(), d, L) == [], "nothing changed: everything must be skipped"
    assert run_pipeline(st(lr=0.2), d, L) == ["train", "evaluate"], "param change must rerun that stage and downstream"
    open(P("raw.txt"), "w").write("3 1 4 1 5 9 2 6 5")
    assert run_pipeline(st(lr=0.2), d, L) == ["prep", "train", "evaluate"]
    os.remove(P("metrics.json"))
    assert run_pipeline(st(lr=0.2), d, L) == ["evaluate"], "a missing output must be rebuilt"
    open(P("raw.txt"), "w").write("3 1 4 1 5 9 2 6 5 ")
    ran = run_pipeline(st(lr=0.2), d, L)
    assert ran == ["prep"], f"{ran}: clean.txt did not change, so downstream stages must be skipped"
    cyc = [dict(name="a", fn=None, deps=["b.txt"], outs=["a.txt"], params={}), dict(name="b", fn=None, deps=["a.txt"], outs=["b.txt"], params={})]
    try: run_pipeline(cyc, d, P("l2.json")); assert False, "cycle not detected"
    except ValueError: pass
    print("PASS")
_check1()

**Written (1a):** In the last step `raw.txt` changes only by a trailing space, `prep` reruns, but does `clean.txt` change? If not, should `train` rerun? What does DVC do here, and why is content hashing better than timestamps (what `make` uses)?

**Your answer:**

## Exercise 2: An experiment tracker (what MLflow does)

`Tracker(root)`:
- `start_run(params, tags=None)` creates `root/<run_id>/` with `params.json`, returns `run_id` (sortable, unique: timestamp + random suffix).
- `log_metric(run_id, key, value, step)` appends to `metrics.jsonl` (one JSON per line, crash-safe: open, append, close).
- `end_run(run_id, status="FINISHED")`.
- `history(run_id, key)` → list of `(step, value)` sorted by step.
- `search(filter_fn=None, order_by=None, mode="max")` → list of run_ids, where `order_by` is a metric key using each run's **last** logged value; runs without that metric go last. `filter_fn(params, status)` filters.

In [ ]:
class Tracker:
    def __init__(self, root): raise NotImplementedError
    def start_run(self, params, tags=None): raise NotImplementedError
    def log_metric(self, run_id, key, value, step): raise NotImplementedError
    def end_run(self, run_id, status="FINISHED"): raise NotImplementedError
    def history(self, run_id, key): raise NotImplementedError
    def search(self, filter_fn=None, order_by=None, mode="max"): raise NotImplementedError

In [ ]:
# ---- CHECK 2 (do not edit) ----
def _check2():
    t = Tracker(tempfile.mkdtemp()); ids = []
    for lr, final in ((1e-3, .71), (3e-3, .78), (1e-2, .64), (3e-2, None)):
        r = t.start_run({"lr": lr}); ids.append(r)
        for s in (30, 10, 20):
            if final is not None: t.log_metric(r, "val_acc", final - (30 - s) / 100, s)
        t.log_metric(r, "loss", 1.0, 0); t.end_run(r, "FAILED" if final is None else "FINISHED")
    assert len(set(ids)) == 4
    h = t.history(ids[1], "val_acc"); assert [x[0] for x in h] == [10, 20, 30] and np.allclose([x[1] for x in h], [.58, .68, .78])
    assert t.search(order_by="val_acc")[:3] == [ids[1], ids[0], ids[2]] and t.search(order_by="val_acc")[3] == ids[3]
    assert t.search(order_by="val_acc", mode="min")[0] == ids[2]
    assert t.search(filter_fn=lambda p, s: s == "FINISHED" and p["lr"] < 5e-3) == ids[:2]
    print("PASS")
_check2()

## Exercise 3: Tests that catch real ML bugs

Write test functions that take a **model factory** (returns a fresh `nn.Module` mapping `(B, 10)` floats to `(B, 3)` logits) and raise `AssertionError` on failure:
- `test_output_shape(factory)`
- `test_overfit_one_batch(factory)`: 8 examples, Adam 1e-2, 300 steps, loss must go below 0.05
- `test_batch_independence(factory)`: in eval mode, the output for example 0 must not depend on the other examples in the batch (catches accidental mixing across the batch dimension)
- `test_finite_on_extreme_inputs(factory)`: inputs ×1e4 must give finite logits

The check runs your tests against one correct model (all must pass) and three buggy ones (each must be caught by at least one test). This is "mutation testing" for your test suite.

In [ ]:
def test_output_shape(factory): raise NotImplementedError
def test_overfit_one_batch(factory): raise NotImplementedError
def test_batch_independence(factory): raise NotImplementedError
def test_finite_on_extreme_inputs(factory): raise NotImplementedError
TESTS = [test_output_shape, test_overfit_one_batch, test_batch_independence, test_finite_on_extreme_inputs]

In [ ]:
# ---- CHECK 3 (do not edit) ----
def _check3():
    nn = torch.nn
    class Good(nn.Module):
        def __init__(s): super().__init__(); s.n = nn.LayerNorm(10); s.f = nn.Sequential(nn.Linear(10, 64), nn.ReLU(), nn.Linear(64, 3))
        def forward(s, x): return s.f(s.n(x))
    class MixesBatch(Good):
        def forward(s, x): return s.f(s.n(x) - s.n(x).mean(0, keepdim=True) * 0.5)
    class FrozenGrad(Good):
        def forward(s, x): return s.f(s.n(x)).detach() + 0 * s.f[2].bias
    class Overflows(nn.Module):
        def __init__(s): super().__init__(); s.f = nn.Sequential(nn.Linear(10, 64), nn.ReLU(), nn.Linear(64, 3))
        def forward(s, x): return s.f(x).exp().exp()
    def outcome(fac):
        fails = []
        for t in TESTS:
            try: t(fac)
            except AssertionError: fails.append(t.__name__)
            except Exception as e: fails.append(t.__name__)
        return fails
    assert outcome(Good) == [], f"your tests fail a correct model: {outcome(Good)}"
    for bug in (MixesBatch, FrozenGrad, Overflows):
        assert outcome(bug), f"no test caught {bug.__name__}"
    print("PASS  caught:", {b.__name__: outcome(b) for b in (MixesBatch, FrozenGrad, Overflows)})
_check3()

## Exercise 4: A CI workflow

Write a GitHub Actions workflow as a YAML string `WORKFLOW` that:
- runs on `pull_request` and on `push` to `main`
- has a job `test` on `ubuntu-latest` with a Python version matrix of `3.11` and `3.12`
- checks out the code, sets up Python with pip caching, installs with `pip install -e .[dev]`, runs `ruff check .` and `pytest -q`
- has a second job `smoke-train` that **needs** `test` and runs `python -m yourpkg.train --steps 20 --device cpu`

In [ ]:
WORKFLOW = '''
# YOUR YAML HERE
'''

In [ ]:
# ---- CHECK 4 (do not edit) ----
def _check4():
    w = yaml.safe_load(WORKFLOW); on = w.get("on", w.get(True))
    assert "pull_request" in on and on["push"]["branches"] == ["main"], "triggers"
    t = w["jobs"]["test"]; assert t["runs-on"] == "ubuntu-latest"
    assert [str(v) for v in t["strategy"]["matrix"]["python-version"]] == ["3.11", "3.12"], "quote the versions: 3.10 as a YAML float becomes 3.1"
    steps = t["steps"]; uses = [s.get("uses", "") for s in steps]; runs = " ".join(s.get("run", "") for s in steps)
    assert any(u.startswith("actions/checkout") for u in uses) and any(u.startswith("actions/setup-python") for u in uses)
    sp = next(s for s in steps if s.get("uses", "").startswith("actions/setup-python"))
    assert sp["with"].get("cache") == "pip" and "matrix.python-version" in str(sp["with"]["python-version"])
    for cmd in ("pip install -e .[dev]", "ruff check .", "pytest -q"): assert cmd in runs, cmd
    s = w["jobs"]["smoke-train"]; assert s["needs"] in ("test", ["test"])
    assert "--steps 20" in " ".join(x.get("run", "") for x in s["steps"])
    print("PASS")
_check4()

## Exercise 5 (laptop + GitHub): the real tools on your Step 3 code

1. Turn your Step 3 code into an installable package (`pyproject.toml`, `src/yourpkg/`), move the Exercise 3 tests into `tests/`, add `pre-commit` with ruff.
2. `dvc init`; define `dvc.yaml` stages download → tokenize → train (tiny config) → evaluate, with `params.yaml`. Add a free DagsHub remote; `dvc push`. Verify `dvc repro` on a fresh clone reproduces `metrics.json`.
3. MLflow: log params, metrics per step and the model as an artifact; register the best model. Use DagsHub's free MLflow server or a local `mlflow ui`.
4. Commit Exercise 4's workflow as `.github/workflows/ci.yml`. Open a PR that breaks a test and confirm CI blocks it.

**Done when** a stranger can clone the repo and reproduce your headline number following only the README.

---
### Hints for self-grading
- 1a: `clean.txt` is identical (split() ignores the space), so a content-addressed system re-runs `prep` but skips `train` and `evaluate`: their dependency hashes did not change. DVC behaves the same way. Timestamps would rerun everything because the file was touched.